## Setup

Session-wide display settings: show all columns and format floats as plain decimals 
instead of scientific notation.

In [1]:
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', None)

## Patients Table

Loading and validating `patients.csv`, the base demographic table for the 
synthetic population. Name fields (`first`, `middle`, `last`, `prefix`, 
`suffix`, `maiden`) are retained here to mirror what a real EMR extract 
would include, but are intentionally excluded from all downstream 
analytical tables in SQL to avoid carrying identifying fields into the 
analysis layer.

In [2]:
patients = pd.read_csv("output/csv/patients.csv")

patients.columns = patients.columns.str.lower()
patients['birthdate'] = pd.to_datetime(patients['birthdate'])
patients['deathdate'] = pd.to_datetime(patients['deathdate'])
patients['zip'] = patients['zip'].astype(str).str.zfill(5)
patients['fips'] = patients['fips'].astype(str).str.zfill(5)

print(f"Shape: {patients.shape}")
patients.head()

Shape: (17739, 28)


,id,birthdate,deathdate,ssn,drivers,passport,prefix,first,middle,last,suffix,maiden,marital,race,ethnicity,gender,birthplace,address,city,state,county,fips,zip,lat,lon,healthcare_expenses,healthcare_coverage,income
0,698c3cd3-5a43-b0bf-c721-253f90fe53ed,1966-02-16,1970-02-14,999-92-7065,NaN,NaN,NaN,Gene733,Deangelo7,Swift555,NaN,NaN,NaN,white,nonhispanic,M,Bolton Massachusetts US,1097 Wiza Corner,Haverhill,Massachusetts,Essex County,25009.0,01835,42.83,-71.05,"12,441.27",0.00,89947
1,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,2011-12-13,NaT,999-14-4411,NaN,NaN,NaN,Logan497,Lewis216,Welch179,NaN,NaN,NaN,white,nonhispanic,M,Lynn Massachusetts US,752 Brown Terrace Apt 50,East Bridgewater,Massachusetts,Plymouth County,NaN,00000,42.01,-70.95,"41,313.97","3,060.77",165705
2,30604641-a5f8-12c0-ad84-9f7d6c737d8a,1969-04-13,1986-07-22,999-10-3048,S99974670,NaN,NaN,Leon728,Lucas404,Marquardt819,NaN,NaN,NaN,asian,hispanic,M,Dedham Massachusetts US,1092 Konopelski Approach,Springfield,Massachusetts,Hampden County,25013.0,01104,42.12,-72.54,"4,798.91","60,020.82",24042
3,5a6c1718-68bd-3bc4-6089-4008c403689f,1969-03-11,2015-05-09,999-78-4979,S99957202,X24149173X,Mr.,Florencio463,Ryan260,Hickle134,NaN,NaN,M,white,nonhispanic,M,West Yarmouth Massachusetts US,985 Lubowitz Plaza,Boston,Massachusetts,Suffolk County,25025.0,02115,42.37,-71.08,"119,572.04","2,370.94",111616
4,812bc940-434f-0195-ddfb-3407d4f0e0fb,1991-10-08,NaT,999-86-7504,S99968634,X6157825X,Mr.,Pablo44,Reinaldo138,Heathcote539,NaN,NaN,M,black,nonhispanic,M,Needham Massachusetts US,679 Ernser Mews,Easthampton,Massachusetts,Hampshire County,25015.0,01027,42.27,-72.65,"102,230.85","16,334.55",63358


### Known Data Limitation: Missing ZIP Codes

Approximately 4,547 patients (~26%) had a ZIP code of `00000`. 
Investigation confirmed all affected patients are legitimately located 
in Massachusetts, with fully populated city/state fields — the missing 
ZIPs cluster around specific real towns (e.g. Amherst, Billerica, 
Natick), pointing to a gap in Synthea's internal city-to-ZIP reference 
table rather than a data error. These `00000` placeholders have been 
converted to nulls to make the missing data explicit.

In [3]:
patients.loc[patients['zip'] == '00000', 'zip'] = None
print(f"Missing zips (converted to null): {patients['zip'].isnull().sum()}")

Missing zips (converted to null): 4547


### Data Quality Checks

Confirming no unexpected nulls (`maiden` is expected to be entirely null 
given the male-only cohort generated), no duplicate patient IDs, and 
sensible ranges on key numeric fields.

In [4]:
print("Nulls per column:")
print(patients.isnull().sum())

print(f"\nNon-null maiden names (expected ~0 given male-only cohort): {patients['maiden'].notnull().sum()}")

print(f"\nDuplicate patient IDs: {patients['id'].duplicated().sum()}")

Nulls per column:
id                         0
birthdate                  0
deathdate              15000
ssn                        0
drivers                 2965
passport                3833
prefix                  3397
first                      0
middle                  3484
last                       0
suffix                 17555
maiden                 17739
marital                 5600
race                       0
ethnicity                  0
gender                     0
birthplace                 0
address                    0
city                       0
state                      0
county                     0
fips                    4547
zip                     4547
lat                        0
lon                        0
healthcare_expenses        0
healthcare_coverage        0
income                     0
dtype: int64

Non-null maiden names (expected ~0 given male-only cohort): 0

Duplicate patient IDs: 0


In [5]:
patients[['income', 'healthcare_expenses', 'healthcare_coverage']].describe()

,income,healthcare_expenses,healthcare_coverage
count,"17,739.00","17,739.00","17,739.00"
mean,"115,324.44","109,675.80","218,145.88"
std,"166,661.28","122,162.42","411,062.53"
min,3.00,100.00,0.00
25%,"31,270.50","20,873.71","11,839.97"
50%,"68,873.00","78,836.12","62,515.32"
75%,"125,514.00","153,000.38","250,676.61"
max,"997,688.00","2,163,948.12","5,081,196.37"


## Conditions Table

Loading and validating `conditions.csv`, which contains every diagnosis 
recorded across all patient encounters. This table is what we'll use to 
identify the prostate cancer cohort.

In [6]:
conditions = pd.read_csv("output/csv/conditions.csv")

conditions.columns = conditions.columns.str.lower()
conditions['start'] = pd.to_datetime(conditions['start'])
conditions['stop'] = pd.to_datetime(conditions['stop'])
conditions['code'] = conditions['code'].astype(str)

print(f"Shape: {conditions.shape}")
conditions.head()

Shape: (653507, 7)


,start,stop,patient,encounter,system,code,description
0,1966-02-16,1966-07-27,698c3cd3-5a43-b0bf-c721-253f90fe53ed,698c3cd3-5a43-b0bf-a097-911f1801540e,http://snomed.info/sct,314529007,Medication review due (situation)
1,2014-11-18,2021-12-21,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,751487cb-067f-0a4d-b3dd-7b115b8f0d5a,http://snomed.info/sct,314529007,Medication review due (situation)
2,2016-09-15,2016-11-12,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,751487cb-067f-0a4d-ae01-d1be023938bc,http://snomed.info/sct,307731004,Injury of tendon of the rotator cuff of should...
3,2016-10-28,2016-11-07,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,751487cb-067f-0a4d-cafa-839d12114387,http://snomed.info/sct,43878008,Streptococcal sore throat (disorder)
4,1966-06-08,1966-06-18,698c3cd3-5a43-b0bf-c721-253f90fe53ed,698c3cd3-5a43-b0bf-61fe-f04ae991989e,http://snomed.info/sct,10509002,Acute bronchitis (disorder)


### Data Quality Checks

Confirming no unexpected nulls, no fully duplicated rows, and that every 
`patient` value has a matching record in the patients table.

In [7]:
print("Nulls per column:")
print(conditions.isnull().sum())

print(f"\nFully duplicated rows: {conditions.duplicated().sum()}")

orphaned = ~conditions['patient'].isin(patients['id'])
print(f"Conditions with no matching patient: {orphaned.sum()}")

Nulls per column:
start               0
stop           166856
patient             0
encounter           0
system              0
code                0
description         0
dtype: int64

Fully duplicated rows: 0
Conditions with no matching patient: 0


## Medications Table

Loading and validating `medications.csv`, which contains every medication
recorded across all patient encounters. This table will be used to identify 
restricted prior medications and chemo treatments. 

In [8]:
medications = pd.read_csv("output/csv/medications.csv")

medications.columns = medications.columns.str.lower()
medications['start'] = pd.to_datetime(medications['start'])
medications['stop'] = pd.to_datetime(medications['stop'])
medications['code'] = medications['code'].astype(str)
medications['reasoncode'] = medications['reasoncode'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

print(f"Shape: {medications.shape}")
medications.head()

Shape: (1013903, 13)


,start,stop,patient,payer,encounter,code,description,base_cost,payer_coverage,dispenses,totalcost,reasoncode,reasondescription
0,2016-10-28 14:14:33+00:00,2016-11-07 18:14:33+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,751487cb-067f-0a4d-cafa-839d12114387,834061,Penicillin V Potassium 250 MG Oral Tablet,370.48,0.00,1,370.48,43878008,Streptococcal sore throat (disorder)
1,1966-06-08 19:19:55+00:00,1966-06-18 19:19:55+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,e03e23c9-4df1-3eb6-a62d-f70f02301496,698c3cd3-5a43-b0bf-61fe-f04ae991989e,313782,Acetaminophen 325 MG Oral Tablet,79.83,0.00,1,79.83,10509002,Acute bronchitis (disorder)
2,2016-11-29 10:10:11+00:00,2016-11-29 10:10:11+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,751487cb-067f-0a4d-2045-91e42be85de7,1535362,sodium fluoride 0.0272 MG/MG Oral Gel,129.94,77.96,1,129.94,103697008,Patient referral for dental care (procedure)
3,1968-02-12 04:57:51+00:00,1968-02-24 04:57:51+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,e03e23c9-4df1-3eb6-a62d-f70f02301496,698c3cd3-5a43-b0bf-8216-0dd3f5d054ee,562251,Amoxicillin 250 MG / Clavulanate 125 MG Oral T...,76.78,0.00,1,76.78,444814009,Viral sinusitis (disorder)
4,1969-04-01 18:57:51+00:00,1969-04-15 18:57:51+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,e03e23c9-4df1-3eb6-a62d-f70f02301496,698c3cd3-5a43-b0bf-e58e-8578b5e9dab5,1652673,Doxycycline Monohydrate 50 MG Oral Tablet,973.03,0.00,1,973.03,NaN,NaN


### Data Quality Checks

Confirming nulls are limited to expected fields (e.g. `stop` for ongoing 
medications, `reasoncode`/`reasondescription` for medications without a 
documented reason), no fully duplicated rows, and full referential 
integrity against the patients table.

In [9]:
print("Nulls per column:")
print(medications.isnull().sum())

print(f"\nFully duplicated rows: {medications.duplicated().sum()}")

orphaned = ~medications['patient'].isin(patients['id'])
print(f"Medications with no matching patient: {orphaned.sum()}")

Nulls per column:
start                     0
stop                  50807
patient                   0
payer                     0
encounter                 0
code                      0
description               0
base_cost                 0
payer_coverage            0
dispenses                 0
totalcost                 0
reasoncode           150094
reasondescription    150094
dtype: int64

Fully duplicated rows: 0
Medications with no matching patient: 0


### Cost Field Sanity Check

Checking for implausible values (negative costs, broken ranges) in the 
financial columns.

In [10]:
medications[['base_cost', 'payer_coverage', 'totalcost']].describe()

,base_cost,payer_coverage,totalcost
count,"1,013,903.00","1,013,903.00","1,013,903.00"
mean,99.14,63.58,"2,203.56"
std,"1,133.12",874.67,"168,720.22"
min,0.00,0.00,0.00
25%,0.91,0.00,1.82
50%,29.48,1.10,30.67
75%,129.94,94.94,129.94
max,"103,958.40","103,958.40","42,311,068.80"


## Procedures Table

Loading and validating `procedures.csv`, which contains every procedure 
performed across all patient encounters.

In [11]:
procedures = pd.read_csv("output/csv/procedures.csv")

procedures.columns = procedures.columns.str.lower()
procedures['start'] = pd.to_datetime(procedures['start'])
procedures['stop'] = pd.to_datetime(procedures['stop'])
procedures['code'] = procedures['code'].astype(str)
procedures['reasoncode'] = procedures['reasoncode'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

print(f"Shape: {procedures.shape}")
procedures.head()

Shape: (2806289, 10)


,start,stop,patient,encounter,system,code,description,base_cost,reasoncode,reasondescription
0,2016-10-28 14:14:33+00:00,2016-10-28 14:29:33+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,751487cb-067f-0a4d-cafa-839d12114387,http://snomed.info/sct,117015009,Throat culture (procedure),"1,932.67",43878008,Streptococcal sore throat (disorder)
1,1966-06-08 18:57:51+00:00,1966-06-08 19:19:55+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,698c3cd3-5a43-b0bf-61fe-f04ae991989e,http://snomed.info/sct,23426006,Measurement of respiratory function (procedure),690.24,10509002,Acute bronchitis (disorder)
2,2016-11-22 08:14:33+00:00,2016-11-22 08:25:32+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,751487cb-067f-0a4d-3075-a44111a269af,http://snomed.info/sct,103697008,Patient referral for dental care (procedure),431.40,NaN,NaN
3,1966-07-27 18:57:51+00:00,1966-07-27 19:12:51+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,698c3cd3-5a43-b0bf-e08b-3e862c18781e,http://snomed.info/sct,430193006,Medication reconciliation (procedure),860.90,NaN,NaN
4,1966-10-26 18:57:51+00:00,1966-10-26 19:12:51+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,698c3cd3-5a43-b0bf-da05-edfb07e1de27,http://snomed.info/sct,430193006,Medication reconciliation (procedure),413.07,NaN,NaN


### Data Quality Checks

In [12]:
print("Nulls per column:")
print(procedures.isnull().sum())

print(f"\nFully duplicated rows: {procedures.duplicated().sum()}")

orphaned = ~procedures['patient'].isin(patients['id'])
print(f"Procedures with no matching patient: {orphaned.sum()}")

Nulls per column:
start                      0
stop                       0
patient                    0
encounter                  0
system                     0
code                       0
description                0
base_cost                  0
reasoncode           1592914
reasondescription    1592914
dtype: int64

Fully duplicated rows: 0
Procedures with no matching patient: 0


### Cost Field Sanity Check

In [13]:
procedures[['base_cost']].describe()

,base_cost
count,"2,806,289.00"
mean,865.61
std,"2,386.19"
min,0.32
25%,431.40
50%,431.40
75%,431.40
max,"267,860.97"


## Encounters Table

Loading and validating `encounters.csv`, which contains every healthcare 
encounter/visit across the patient population. This is expected to be the 
largest table, since each patient can accumulate many encounters over 
their simulated lifetime.

In [14]:
encounters = pd.read_csv("output/csv/encounters.csv")

encounters.columns = encounters.columns.str.lower()
encounters['start'] = pd.to_datetime(encounters['start'])
encounters['stop'] = pd.to_datetime(encounters['stop'])
encounters['code'] = encounters['code'].astype(str)
encounters['reasoncode'] = encounters['reasoncode'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

print(f"Shape: {encounters.shape}")
encounters.head()

Shape: (1008929, 15)


,id,start,stop,patient,organization,provider,payer,encounterclass,code,description,base_encounter_cost,total_claim_cost,payer_coverage,reasoncode,reasondescription
0,698c3cd3-5a43-b0bf-a097-911f1801540e,1966-02-16 18:57:51+00:00,1966-02-16 19:12:51+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,83a08dcf-7ed6-3d2e-afc0-a72c05d2563c,311c0898-b13d-32b8-b03b-7fc6c4da003e,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,410620009,Well child visit (procedure),136.80,211.38,0.00,NaN,NaN
1,751487cb-067f-0a4d-b3dd-7b115b8f0d5a,2014-11-18 08:14:33+00:00,2014-11-18 08:29:33+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,b44955d1-f0e2-3beb-a013-708a81dbe430,6f994348-2304-3973-ae0a-9e23f7e94bc0,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,wellness,410620009,Well child visit (procedure),136.80,704.20,0.00,NaN,NaN
2,751487cb-067f-0a4d-ae01-d1be023938bc,2016-09-15 08:14:33+00:00,2016-09-15 08:29:33+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,51370692-6296-3150-8672-559fc73f964f,5e454349-db44-3f9b-868d-2720945ae819,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,ambulatory,185347001,Encounter for problem (procedure),85.55,85.55,0.00,307731004,Injury of tendon of the rotator cuff of should...
3,751487cb-067f-0a4d-cafa-839d12114387,2016-10-28 14:14:33+00:00,2016-10-28 14:29:33+00:00,751487cb-067f-0a4d-3ac4-7dafa2e1d50d,51370692-6296-3150-8672-559fc73f964f,5e454349-db44-3f9b-868d-2720945ae819,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,ambulatory,185345009,Encounter for symptom (procedure),85.55,"2,018.22",0.00,43878008,Streptococcal sore throat (disorder)
4,698c3cd3-5a43-b0bf-cf27-ab3a1750dbb7,1966-03-23 18:57:51+00:00,1966-03-23 19:12:51+00:00,698c3cd3-5a43-b0bf-c721-253f90fe53ed,83a08dcf-7ed6-3d2e-afc0-a72c05d2563c,311c0898-b13d-32b8-b03b-7fc6c4da003e,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,410620009,Well child visit (procedure),136.80,136.80,0.00,NaN,NaN


### Data Quality Checks

In [15]:
print("Nulls per column:")
print(encounters.isnull().sum())

print(f"\nDuplicate encounter IDs: {encounters['id'].duplicated().sum()}")

orphaned = ~encounters['patient'].isin(patients['id'])
print(f"Encounters with no matching patient: {orphaned.sum()}")

Nulls per column:
id                          0
start                       0
stop                        0
patient                     0
organization                0
provider                    0
payer                       0
encounterclass              0
code                        0
description                 0
base_encounter_cost         0
total_claim_cost            0
payer_coverage              0
reasoncode             399413
reasondescription      399413
dtype: int64

Duplicate encounter IDs: 0
Encounters with no matching patient: 0


### Encounter Class Distribution

A quick look at the types of encounters present (e.g. ambulatory, 
emergency, inpatient, wellness) as a sanity check that the values are 
sensible categories rather than free text or garbage values.

In [16]:
encounters['encounterclass'].value_counts()

encounterclass
ambulatory    573297
wellness      220146
outpatient     93836
urgentcare     50857
emergency      36522
inpatient      20087
home            5804
virtual         3026
hospice         2901
snf             2453
Name: count, dtype: int64

## Organizations Table

Loading and validating `organizations.csv`, which contains the healthcare 
organizations/facilities patients were seen at.

In [17]:
organizations = pd.read_csv("output/csv/organizations.csv")

def fix_ZIP(z):
    if pd.isnull(z):
        return z
    z = str(int(z))  # strip any decimal artifacts first
    if len(z) <= 5:
        return z.zfill(5)
    else:
        return z.zfill(9)

organizations.columns = organizations.columns.str.lower()
organizations['zip'] = organizations['zip'].apply(fix_ZIP)
organizations['zip'] = organizations.apply(
    lambda row: row['zip'][:5] + '-' + row['zip'][5:] if len(row['zip']) == 9 else row['zip'],
    axis=1
)
print(f"Shape: {organizations.shape}")
organizations.head()

Shape: (1174, 11)


,id,name,address,city,state,zip,lat,lon,phone,revenue,utilization
0,74ab949d-17ac-3309-83a0-13b4405c66aa,Fitchburg Outpatient Clinic,881 Main Street,Fitchburg,MA,01420,42.59,-71.81,978-342-9781 Or 978-342-9781,0.00,48297
1,da92d3fc-5445-3825-a937-043ef0d6ecd0,TRINITY HOME CARE INC.,336 GRATTAN ST,CHICOPEE,MA,01020-1314,42.17,-72.59,4033313294,0.00,52
2,9d7fb5a1-bf03-3928-a8b3-a2fceba4bd2c,CARING HEARTS HOMECARE INC,188 MAIN ST,WILMINGTON,MA,01887-2046,42.56,-71.18,9786585104,0.00,20
3,588f6ce6-b8db-3588-8189-29db2680a313,BOSTON HEALTH CARE FOR THE HOMELESS PROGRAM INC,461 WALNUT AVE,JAMAICA PLAIN,MA,02130-2331,42.31,-71.10,8576541550,0.00,818
4,324b4137-57a0-3ae0-89db-1c33f57ae0c1,LOWN ACQUISITION LLC,134 NORTH ST,NORTH READING,MA,01864-1315,42.59,-71.11,7818262393,0.00,21


### Data Quality Checks

In [18]:
print("Nulls per column:")
print(organizations.isnull().sum())

print(f"\nDuplicate organization IDs: {organizations['id'].duplicated().sum()}")

Nulls per column:
id             0
name           0
address        0
city           0
state          0
zip            0
lat            0
lon            0
phone          0
revenue        0
utilization    0
dtype: int64

Duplicate organization IDs: 0


## Summary

All six tables (`patients`, `conditions`, `medications`, `procedures`, 
`encounters`, `organizations`) have been loaded, type-corrected, and 
validated for nulls, duplicates, and referential integrity. No orphaned 
records were found in any child table, and no unexpected data quality 
issues were identified beyond the type corrections applied above 
(dates parsed from strings, zip/code fields converted from numeric to 
text to preserve leading zeros).

Cleaned data is ready to be loaded into PostgreSQL for cohort 
identification and eligibility analysis in SQL.

## Load Data into PostgreSQL

In [19]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

username = "postgres"
password = os.getenv("DB_PASSWORD")
host = "localhost"
port = "5432"
database = "prostate_trial_eligibility"

engine = create_engine(f"postgresql://{username}:{password}@{host}:{port}/{database}")
connection = engine.connect()
print("Connection successful!")
connection.close()

Connection successful!


In [20]:
patients.to_sql('patients', engine, if_exists='append', index=False)
print("Patients loaded successfully")

Patients loaded successfully


In [21]:
conditions.to_sql('conditions', engine, if_exists='append', index=False)
print("Conditions loaded successfully")

Conditions loaded successfully


In [22]:
medications.to_sql('medications', engine, if_exists='append', index=False)
print("Medications loaded successfully")

Medications loaded successfully


In [23]:
procedures.to_sql('procedures', engine, if_exists='append', index=False)
print("Procedures loaded successfully")

Procedures loaded successfully


In [24]:
organizations.to_sql('organizations', engine, if_exists='append', index=False)
print("Organizations loaded successfully")

Organizations loaded successfully


In [25]:
encounters.to_sql('encounters', engine, if_exists='append', index=False)
print("Encounters loaded successfully")

Encounters loaded successfully
